# MP02 — NumPy Image Laboratory
Implementation and a real training-crop experiment are executed; learner mastery remains unassessed.

Objective: indexing, slicing, reshaping, broadcasting, normalization, and vectorized operations. Read the [original brief](../stage_01_python_numpy/MP02_numpy_image_laboratory/README.md).

## Arrays before images
An array stores numbers in indexed positions. We use `image[row, column, band]`: H rows, W columns, C bands, shape `(H,W,C)`. Indexing starts at zero; slicing excludes its stop. A reshape changes the grouping of the same ordered values; a transpose changes axis order. These are different operations.

For a 1×2×3 array `[[[250,5,10],[1,100,255]]]`, adding channel offsets `[20,-10,2]` should give `[[[255,0,12],[21,90,255]]]` after clipping to [0,255]. A length-C vector broadcasts over H and W: the same offset applies at every pixel. Convert uint8 to floating point *before* addition, because uint8 cannot represent numbers above 255 or negative values.

For channel c, brightness is y[h,w,c] = min(M,max(0,x[h,w,c]+d[c])). M is the maximum allowed value; d is an offset in the same units as x. This operation illustrates arrays; it is not sensor calibration.

For a boolean H×W validity mask m and K valid pixels, mean[c] = sum(x[h,w,c] over valid pixels)/K. Population variance[c] = sum((x[h,w,c]-mean[c])² over valid pixels)/K. Variance has squared units. We use K, not K−1, because we describe the selected pixels rather than estimate an unbiased population variance. Empty masks are errors, not zero-valued observations.


In [ ]:
from pathlib import Path
import sys
root=Path.cwd()
if not (root/'src').exists(): root=root.parent
sys.path.insert(0,str(root/'src'))
import numpy as np
from rssr.array_lab import brighten,masked_channel_stats
a=np.array([[[250,5,10],[1,100,255]]],dtype=np.uint8)
expected=np.array([[[255,0,12],[21,90,255]]])
np.testing.assert_array_equal(brighten(a,[20,-10,2]),expected)
np.testing.assert_array_equal(brighten(a,[20,-10,2],loop=True),expected)
print(brighten(a,[20,-10,2]))
print('uint8 wrap example:', np.add(np.uint8(250),np.uint8(20)))

## Index, slice, reshape, transpose
Predict each shape before running. Selecting one band removes an axis; selecting a one-band slice preserves it. A basic slice can share memory with its parent, so copy before changing a crop when the original must remain intact.

In [ ]:
grid=np.arange(24).reshape(2,4,3)
print('band:',grid[:,:,0].shape,'one-band slice:',grid[:,:,0:1].shape)
chw=grid.transpose(2,0,1)
assert chw.shape==(3,2,4) and chw[2,1,3]==grid[1,3,2]
crop=grid[:,1:3].copy(); crop[:]=0
assert grid[0,1,0]==3
pixels=grid.reshape(-1,3)
print('pixel table:',pixels.shape)

## Normalization and masks
For uint8 display data, x/255 maps to [0,1]. For scientific data, use declared units and calibration. Dividing each image by its own maximum changes the meaning of equal physical values across scenes. The pilot uses a fixed divisor of 10000 for numerical scaling; physical calibration is unverified.

In the example below, channel values [1,3,5] have mean 3 and variance (4+0+4)/3 = 8/3. The invalid pixel must not contribute.

In [ ]:
x=np.array([[[1,2],[3,4]],[[5,6],[999,999]]],dtype=float)
mask=np.array([[True,True],[True,False]])
mean,var=masked_channel_stats(x,mask)
np.testing.assert_allclose(mean,[3,4]); np.testing.assert_allclose(var,[8/3,8/3])
print('mean:',mean,'variance:',var)
print('centered valid-pixel mean:',(x[mask]-mean).mean(axis=0))

## Real-data experiment and observed evidence
Run `python scripts/mp02_experiment.py` from the repository root after the pinned pilot download. It checks the source checksum, reads the upper-left 64×64 crop from the **training** NAIP scene, preserves all four bands and valid-pixel masks, and compares explicit loops with NumPy operations before timing. It does not read the test scene.

The experiment writes [mp02-array-lab.json](../reports/mp02-array-lab.json). The original executed run used all 4096 valid pixels. Brightness matched exactly; masked means/variances matched within 1e-12. The next cell reads recorded results, not a new timing run. One warm-up and five repetitions were used; medians include input validation/allocation and exclude raster IO. CPU contention and array size affect timings. Vectorization moves iteration into compiled array operations; it does not eliminate computation.


In [ ]:
import json
record_path=root/'reports/mp02-array-lab.json'
if record_path.exists():
    record=json.loads(record_path.read_text())
    for row in record['timings']:
        print(row['operation'],row['implementation'],f"{1000*row['median_seconds']:.4f} ms")
    print(record['limitations'])
else:
    print('Real-data experiment pending in this checkout; run the explicit script.')

## Practice — attempt before reading solutions
1. Normalize the uint8 example to [0,1]. Convert the channel offsets into those units and show that rescaling the result by 255 matches the worked answer.
2. Change the validity mask to select only the top row. Predict each channel mean and variance, then verify.
3. Explain why `reshape(3,2,4)` is not a valid replacement for HWC-to-CHW transpose. Find a differing element.
4. Change one pixel in a view, then repeat with a copy. Explain which original array changes.

## Solution guide — generated examples, not learner submissions
1. Both values and offsets must be divided by 255; the clipping maximum becomes 1. Scaling only the values changes the operation.
2. Valid values are [1,3] and [2,4]; means are [2,3], population variances [1,1].
3. Reshape preserves flat ordering, whereas transpose preserves the semantic relationship between pixel and band coordinates.
4. Basic slicing shares the underlying data; `.copy()` allocates independent storage.


In [ ]:
np.testing.assert_allclose(brighten(a.astype(float)/255,np.array([20,-10,2])/255,1)*255,expected)
new_mask=np.array([[True,True],[False,False]])
m,v=masked_channel_stats(x,new_mask)
np.testing.assert_allclose(m,[2,3]); np.testing.assert_allclose(v,[1,1])
assert not np.array_equal(grid.reshape(3,2,4),grid.transpose(2,0,1))
original=np.arange(6).reshape(2,3); view=original[:,1:]; view[0,0]=99
assert original[0,1]==99
independent=original[:,1:].copy(); independent[0,0]=7
assert original[0,1]==99
print('Solution invariants passed; learner mastery remains unassessed.')

## Reflection and limitations
The executed crop showed the same mathematical outputs for both implementations, with shorter vectorized timings on this machine. Equality comes before speed: a fast calculation with overflow or an incorrect mask is still wrong. This experiment covers one crop, not all array sizes or hardware. Brightness transformations are educational and must not be confused with physical reflectance correction.

Local Jupyter and Colab setup: [setup guide](../docs/setup.md). Colab requires cloning this repository, installing it, and setting its root as the working directory before running. The optional real-data script must use the pinned manifest. Run offline checks with `python -m pytest tests/test_array_lab.py`.

References: [NumPy beginner guide](https://numpy.org/doc/stable/user/absolute_beginners.html), [broadcasting](https://numpy.org/doc/stable/user/basics.broadcasting.html), and the [dataset card](../docs/dataset-cards.md). Implementation validation does not establish that a learner has completed the exercises.